## Extraction

#### Option 1 - JSON with base64 images (self-contained)

In [2]:
import fitz # PyMuPDF
import os
import json
import base64

PDF_DIR = "../data/pdf"
OUTPUT_JSON = "pdf_data.json"

all_docs = []

for filename in os.listdir(PDF_DIR):
    """Extract pdf from directory"""

    if not filename.endswith(".pdf"):
        """ If no PDF available then exit"""
        continue

    path = os.path.join(PDF_DIR, filename)
    doc = fitz.open(path)

    pdf_data = {
        "file_name":filename,
        "pages":[]
    }

    for page_num, page in enumerate(doc):
        """ Iterate each page of the pdf"""
        text = page.get_text() # extract pdf text

        images_data = []
        for img_index, img in enumerate(page.get_images(full=True)):
            """ Extract each image of pages and extract with image index number"""
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]

            image_b64 = base64.b64encode(image_bytes).decode("utf-8")

            images_data.append({
                "image_index":img_index,
                "extension":image_ext,
                "base64":image_b64
            })

        pdf_data["pages"].append({
            "page_number":page_num,
            "text":text,
            "images":images_data
        })

    all_docs.append(pdf_data)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(all_docs, f, ensure_ascii=False, indent=2)

### Option 2 – Save images to disk + reference in JSON (better for large PDFs)

In [ ]:
import fitz
import os
import json

PDF_DIR = "../data/pdf"
IMAGE_DIR = "extracted_images"
OUTPUT_JSON = "pdf_data.json"

os.makedirs(IMAGE_DIR, exist_ok=True)
all_docs = []

for filename in os.listdir(PDF_DIR):
    if not filename.endswith(".pdf"):
        continue

    path = os.path.join(PDF_DIR, filename)
    doc = fitz.open(path)

    pdf_data = {
        "file_name": filename,
        "pages": []
    }

    for page_num, page in enumerate(doc):
        text = page.get_text()
        images_data = []

        for img_index, img in enumerate(page.get_images(full=True)):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]

            image_name = f"{filename}_p{page_num}_{img_index}.{image_ext}"
            image_path = os.path.join(IMAGE_DIR, image_name)

            with open(image_path, "wb") as f:
                f.write(image_bytes)

            images_data.append({
                "image_index": img_index,
                "path": image_path
            })

        pdf_data["pages"].append({
            "page_number": page_num,
            "text": text,
            "images": images_data
        })

    all_docs.append(pdf_data)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(all_docs, f, ensure_ascii=False, indent=2)

print("Saved JSON + images folder")
